In [251]:
import numpy as np
import pandas as pd
import ast  ## used to convert string-list to list
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem.porter import PorterStemmer
from sklearn.metrics.pairwise import cosine_similarity

In [252]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [253]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [254]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [255]:
movies = movies.merge(credits,on='title')

In [256]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

In [257]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [258]:
# genres
# id
# keywords
# title
# overview
# cast
# crew

In [259]:
movies = movies[['movie_id','title','overview','genres','cast','crew','keywords','release_date','budget','vote_average','vote_count']]

In [260]:
movies.head(1)

,movie_id,title,overview,genres,cast,crew,keywords,release_date,budget,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",2009-12-10,237000000,7.2,11800


In [261]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      4809 non-null   int64  
 1   title         4809 non-null   object 
 2   overview      4806 non-null   object 
 3   genres        4809 non-null   object 
 4   cast          4809 non-null   object 
 5   crew          4809 non-null   object 
 6   keywords      4809 non-null   object 
 7   release_date  4808 non-null   object 
 8   budget        4809 non-null   int64  
 9   vote_average  4809 non-null   float64
 10  vote_count    4809 non-null   int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 413.4+ KB


In [262]:
movies.dropna(inplace=True)

In [263]:
movies['release_date'] = pd.to_datetime(movies['release_date'])

In [264]:
movies.dropna(inplace=True)

In [265]:
movies['year'] = (movies['release_date'].dt.year).astype('int32')

In [266]:
movies.head()

,movie_id,title,overview,genres,cast,crew,keywords,release_date,budget,vote_average,vote_count,year
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",2009-12-10,237000000,7.2,11800,2009
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",2007-05-19,300000000,6.9,4500,2007
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",2015-10-26,245000000,6.3,4466,2015
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",2012-07-16,250000000,7.6,9106,2012
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",2012-03-07,260000000,6.1,2124,2012


In [267]:
movies.duplicated().sum()

np.int64(0)

In [268]:
def converter(obj):
    L = []
    l = ast.literal_eval(obj)
    for i in l:
        L.append(i['name'])
    return L

In [269]:
movies['genres'] = movies['genres'].apply(converter)

In [270]:
movies['keywords'] = movies['keywords'].apply(converter)

In [271]:
movies['cast'] = movies['cast'].apply(converter).apply(lambda x:x[:3])

In [272]:
def fetch_director(obj):
    L = []
    l = ast.literal_eval(obj)
    for i in l:
        if i['job']=='Director':
            L.append(i['name'])
        else:
            continue
    return L

In [273]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [274]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [275]:
movies.head(1)

,movie_id,title,overview,genres,cast,crew,keywords,release_date,budget,vote_average,vote_count,year
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[culture clash, future, space war, space colon...",2009-12-10,237000000,7.2,11800,2009


In [276]:
movies.head()

,movie_id,title,overview,genres,cast,crew,keywords,release_date,budget,vote_average,vote_count,year
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[culture clash, future, space war, space colon...",2009-12-10,237000000,7.2,11800,2009
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"[ocean, drug abuse, exotic island, east india ...",2007-05-19,300000000,6.9,4500,2007
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],"[spy, based on novel, secret agent, sequel, mi...",2015-10-26,245000000,6.3,4466,2015
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],"[dc comics, crime fighter, terrorist, secret i...",2012-07-16,250000000,7.6,9106,2012
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],"[based on novel, mars, medallion, space travel...",2012-03-07,260000000,6.1,2124,2012


In [277]:
movies['genres_length'] = movies['genres'].apply(lambda x:len(x))

In [278]:
movies = movies[movies['genres_length'] != 0]

In [279]:
movies['genre'] = movies['genres'].apply(lambda x:x[0])

In [280]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])

In [281]:
movies['tags'] = movies['overview']+movies['cast']+movies['crew']+movies['genres']+movies['keywords']  

In [312]:
new_df = movies[['movie_id','title','tags','genre','release_date','year','vote_average','vote_count']]

In [313]:
new_df.head()

,movie_id,title,tags,genre,release_date,year,vote_average,vote_count
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",Action,2009-12-10,2009,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",Adventure,2007-05-19,2007,6.9,4500
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",Action,2015-10-26,2015,6.3,4466
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",Action,2012-07-16,2012,7.6,9106
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",Action,2012-03-07,2012,6.1,2124


In [314]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4778 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   movie_id      4778 non-null   int64         
 1   title         4778 non-null   object        
 2   tags          4778 non-null   object        
 3   genre         4778 non-null   object        
 4   release_date  4778 non-null   datetime64[ns]
 5   year          4778 non-null   int32         
 6   vote_average  4778 non-null   float64       
 7   vote_count    4778 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(2), object(3)
memory usage: 317.3+ KB


In [315]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\vikas\AppData\Local\Temp\ipykernel_9852\3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [316]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\vikas\AppData\Local\Temp\ipykernel_9852\3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [317]:
new_df.head()

,movie_id,title,tags,genre,release_date,year,vote_average,vote_count
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",Action,2009-12-10,2009,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",Adventure,2007-05-19,2007,6.9,4500
2,206647,Spectre,a cryptic message from bond’s past sends him o...,Action,2015-10-26,2015,6.3,4466
3,49026,The Dark Knight Rises,following the death of district attorney harve...,Action,2012-07-16,2012,7.6,9106
4,49529,John Carter,"john carter is a war-weary, former military ca...",Action,2012-03-07,2012,6.1,2124


In [318]:
cv = CountVectorizer(max_features=5000,stop_words='english')

In [319]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [320]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [321]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [322]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [323]:
ps = PorterStemmer()

In [324]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\vikas\AppData\Local\Temp\ipykernel_9852\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


#### cosine similarity

In [325]:
similarity = cosine_similarity(vectors)

In [326]:
similarity[0]

array([1.        , 0.08980265, 0.05892557, ..., 0.05270463, 0.02512595,
       0.        ])

In [327]:
new_df[new_df['title']=='Avatar'].index[0]

np.int64(0)

In [328]:
def recommend(movie):
    movie_idx = new_df[new_df['genre']==movie].index[0] #index of movie in dataset
    distances = similarity[movie_idx]    # index of similarity matrix
    movies_list = sorted(list(enumerate(distances)),
                         reverse=True,key=lambda x:x[1])[1:6]
    # l = []
    # for i in movies_list:
    #     l.append(new_df.iloc[i[0]].title)

    # return l
    return movies_list

In [329]:
recommend('Action')

[(539, np.float64(0.25724787771376323)),
 (1192, np.float64(0.2545875386086578)),
 (260, np.float64(0.24759378423606915)),
 (1214, np.float64(0.24733283866815065)),
 (507, np.float64(0.24720661623652207))]

In [330]:
import pickle

In [331]:
pickle.dump(new_df,open('movies.pkl','wb'))

In [332]:
pickle.dump(new_df.to_dict(),open('movies_dict.pkl','wb'))

In [333]:
movie_list = pickle.load(open('movies.pkl', 'rb'))

In [334]:
movie_list

,movie_id,title,tags,genre,release_date,year,vote_average,vote_count
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa...",Action,2009-12-10,2009,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c...",Adventure,2007-05-19,2007,6.9,4500
2,206647,Spectre,a cryptic messag from bond’ past send him on a...,Action,2015-10-26,2015,6.3,4466
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...,Action,2012-07-16,2012,7.6,9106
4,49529,John Carter,"john carter is a war-weary, former militari ca...",Action,2012-03-07,2012,6.1,2124
...,...,...,...,...,...,...,...,...
4803,67238,Cavite,"adam, a secur guard, travel from california to...",Foreign,2005-03-12,2005,7.5,2
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...,Action,1992-09-04,1992,6.6,238
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...,Comedy,2011-12-26,2011,5.9,5
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q...",Comedy,2013-10-13,2013,7.0,6


In [335]:
set(movie_list['genre'].tolist())

{'Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Foreign',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western'}

In [336]:
pickle.dump(similarity,open('similarity.pkl','wb'))

In [337]:
new_df.iloc[3].title

'The Dark Knight Rises'

In [338]:
new_df.iloc[11].title

'Quantum of Solace'

In [339]:
movie_list = pickle.load(open('similarity.pkl', 'rb'))

In [340]:
movie_list

array([[1.        , 0.08980265, 0.05892557, ..., 0.05270463, 0.02512595,
        0.        ],
       [0.08980265, 1.        , 0.06350006, ..., 0.        , 0.02707652,
        0.        ],
       [0.05892557, 0.06350006, 1.        , ..., 0.        , 0.02665009,
        0.        ],
       ...,
       [0.05270463, 0.        , 0.        , ..., 1.        , 0.09534626,
        0.        ],
       [0.02512595, 0.02707652, 0.02665009, ..., 0.09534626, 1.        ,
        0.04828045],
       [0.        , 0.        , 0.        , ..., 0.        , 0.04828045,
        1.        ]])

In [341]:
movies['genre'].values

array(['Action', 'Adventure', 'Action', ..., 'Comedy', 'Comedy',
       'Documentary'], dtype=object)